In [ ]:
# Cell 1
!pip -q install timm grad-cam

import os
import gc
import time
import copy
import json
import shutil
import random
import hashlib
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import timm

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
print("DEVICE:", DEVICE)

In [ ]:
# Cell 2
DATA_ROOT = Path("/kaggle/input/datasets/tasminashifa/rice-leaf-disease-images-dataset/Rice Leaf Disease Images")
WORK_ROOT = Path("/kaggle/working/rice_pipeline")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

DUP_CLEAN_ROOT = WORK_ROOT / "dedup_cleaned"
SPLIT_ROOT = WORK_ROOT / "split_data"
CHECKPOINT_ROOT = WORK_ROOT / "checkpoints"
FIG_ROOT = WORK_ROOT / "figures"

CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
FIG_ROOT.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
NUM_CLASSES = 4
TARGET_PER_CLASS = 2500
NUM_WORKERS = 2
PATIENCE = 3
AMP_ENABLED = True if DEVICE == "cuda" else False

MODEL_CONFIGS = {
    "resnet50": {"model_name": "resnet50", "batch_size": 16, "head_epochs": 2, "ft_epochs": 8, "head_lr": 1e-3, "ft_lr": 1e-4, "mode": "full"},
    "densenet121": {"model_name": "densenet121", "batch_size": 16, "head_epochs": 2, "ft_epochs": 8, "head_lr": 1e-3, "ft_lr": 1e-4, "mode": "full"},
    "mobilenetv3_large": {"model_name": "mobilenetv3_large_100", "batch_size": 16, "head_epochs": 2, "ft_epochs": 6, "head_lr": 1e-3, "ft_lr": 1e-4, "mode": "full"},
    "efficientnet_b0": {"model_name": "efficientnet_b0", "batch_size": 16, "head_epochs": 2, "ft_epochs": 8, "head_lr": 1e-3, "ft_lr": 1e-4, "mode": "full"},
    "swin_t": {"model_name": "swin_tiny_patch4_window7_224", "batch_size": 8, "head_epochs": 2, "ft_epochs": 10, "head_lr": 5e-4, "ft_lr": 1e-4, "mode": "partial"},
    "vit_small": {"model_name": "vit_small_patch16_224", "batch_size": 8, "head_epochs": 3, "ft_epochs": 11, "head_lr": 5e-4, "ft_lr": 5e-5, "mode": "partial"},
}

print("DATA_ROOT:", DATA_ROOT)
print("Exists:", DATA_ROOT.exists())
print("\nClass folders:")
for cls_dir in sorted(DATA_ROOT.iterdir()):
    if cls_dir.is_dir():
        print(f" - {cls_dir.name}: {len(list(cls_dir.glob('*')))} files")

In [ ]:
# Cell 3
def count_images_by_class(root: Path):
    counts = {}
    for cls_dir in sorted(root.iterdir()):
        if cls_dir.is_dir():
            counts[cls_dir.name] = len([x for x in cls_dir.iterdir() if x.is_file()])
    return counts

def pretty_print_counts(title, counts):
    print(f"\n{title}")
    for k, v in counts.items():
        print(f" - {k}: {v}")
    print(" - Total:", sum(counts.values()))

def file_md5(path, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def verify_image(path):
    try:
        with Image.open(path) as im:
            im.verify()
        with Image.open(path) as im:
            _ = im.convert("RGB")
        return True
    except Exception:
        return False

raw_counts = count_images_by_class(DATA_ROOT)
pretty_print_counts("Before duplicate removal class distribution", raw_counts)

all_files = []
corrupted = []
for cls_dir in sorted(DATA_ROOT.iterdir()):
    if not cls_dir.is_dir():
        continue
    for img_path in sorted(cls_dir.iterdir()):
        if img_path.is_file():
            if verify_image(img_path):
                all_files.append((cls_dir.name, img_path))
            else:
                corrupted.append((cls_dir.name, str(img_path)))

print("\nCorrupted images found:", len(corrupted))

hash_groups = {}
for cls_name, img_path in all_files:
    digest = file_md5(img_path)
    hash_groups.setdefault((cls_name, digest), []).append(img_path)

duplicate_groups = {k: v for k, v in hash_groups.items() if len(v) > 1}
duplicate_files = sum(len(v) - 1 for v in duplicate_groups.values())

print("Exact duplicate groups found:", len(duplicate_groups))
print("Duplicate files to remove:", duplicate_files)

if duplicate_groups:
    if DUP_CLEAN_ROOT.exists():
        shutil.rmtree(DUP_CLEAN_ROOT)
    DUP_CLEAN_ROOT.mkdir(parents=True, exist_ok=True)
    for cls_dir in sorted(DATA_ROOT.iterdir()):
        if not cls_dir.is_dir():
            continue
        out_cls = DUP_CLEAN_ROOT / cls_dir.name
        out_cls.mkdir(parents=True, exist_ok=True)
        seen_hashes = set()
        for img_path in sorted(cls_dir.iterdir()):
            if not img_path.is_file():
                continue
            if not verify_image(img_path):
                continue
            digest = file_md5(img_path)
            if digest in seen_hashes:
                continue
            seen_hashes.add(digest)
            shutil.copy2(img_path, out_cls / img_path.name)
    SOURCE_ROOT = DUP_CLEAN_ROOT
    print("Duplicate removal applied.")
else:
    SOURCE_ROOT = DATA_ROOT
    print("No duplicate removal needed.")

clean_counts = count_images_by_class(SOURCE_ROOT)
pretty_print_counts("After duplicate removal class distribution", clean_counts)
print("\nSource root for split:", SOURCE_ROOT)

In [ ]:
# Cell 4
def build_dataframe_from_root(root: Path):
    rows = []
    for cls_dir in sorted(root.iterdir()):
        if not cls_dir.is_dir():
            continue
        for img_path in sorted(cls_dir.iterdir()):
            if img_path.is_file():
                rows.append({"class_name": cls_dir.name, "path": str(img_path)})
    return pd.DataFrame(rows)

df = build_dataframe_from_root(SOURCE_ROOT)
print("Usable images after preprocessing:", len(df))
display(df.head())

if SPLIT_ROOT.exists():
    shutil.rmtree(SPLIT_ROOT)
for split_name in ["train", "valid", "test"]:
    for cls_name in sorted(df["class_name"].unique()):
        (SPLIT_ROOT / split_name / cls_name).mkdir(parents=True, exist_ok=True)

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df["class_name"])
valid_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["class_name"])

def copy_split(split_df, split_name):
    for _, row in split_df.iterrows():
        src = Path(row["path"])
        dst = SPLIT_ROOT / split_name / row["class_name"] / src.name
        shutil.copy2(src, dst)

copy_split(train_df, "train")
copy_split(valid_df, "valid")
copy_split(test_df, "test")

def split_counts(frame, name):
    counts = frame["class_name"].value_counts().sort_index().to_dict()
    print(f"\n{name} distribution")
    total = 0
    for k, v in counts.items():
        total += v
        print(f" - {k}: {v}")
    print(f" - Total: {total}")
    return counts

train_counts = split_counts(train_df, "Train")
valid_counts = split_counts(valid_df, "Valid")
test_counts = split_counts(test_df, "Test")

split_table = pd.DataFrame({
    "Classes": sorted(train_counts.keys()),
    "Train": [train_counts[c] for c in sorted(train_counts.keys())],
    "Valid": [valid_counts[c] for c in sorted(valid_counts.keys())],
    "Test": [test_counts[c] for c in sorted(test_counts.keys())],
})
split_table["Total"] = split_table["Train"] + split_table["Valid"] + split_table["Test"]

print("\nPaper dataset table")
display(split_table)

print("\nSplit root:", SPLIT_ROOT)

In [ ]:
# Cell 5
train_root = SPLIT_ROOT / "train"
valid_root = SPLIT_ROOT / "valid"
test_root = SPLIT_ROOT / "test"

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.03),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

base_train_dataset = datasets.ImageFolder(train_root, transform=train_transform)
valid_dataset = datasets.ImageFolder(valid_root, transform=eval_transform)
test_dataset = datasets.ImageFolder(test_root, transform=eval_transform)

idx_to_class = {v: k for k, v in base_train_dataset.class_to_idx.items()}
train_targets = [y for _, y in base_train_dataset.samples]
train_counter = Counter(train_targets)

print("Train class distribution before augmentation")
for idx, cnt in sorted(train_counter.items()):
    print(f" - {idx_to_class[idx]}: {cnt}")
print(" - Total:", sum(train_counter.values()))

print("\nEffective class balance after augmentation target")
for idx in sorted(train_counter.keys()):
    print(f" - {idx_to_class[idx]}: {TARGET_PER_CLASS}")
print(" - Total:", TARGET_PER_CLASS * len(train_counter))

sample_weights = [1.0 / train_counter[y] for y in train_targets]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=TARGET_PER_CLASS * len(train_counter), replacement=True)

def make_loaders(batch_size):
    train_dataset = datasets.ImageFolder(train_root, transform=train_transform)
    valid_ds = datasets.ImageFolder(valid_root, transform=eval_transform)
    test_ds = datasets.ImageFolder(test_root, transform=eval_transform)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, valid_loader, test_loader

print("\nClasses mapping")
print(base_train_dataset.class_to_idx)

In [ ]:
# Cell 6
def build_model(model_key):
    cfg = MODEL_CONFIGS[model_key]
    return timm.create_model(cfg["model_name"], pretrained=True, num_classes=NUM_CLASSES)

def freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False

def unfreeze_all(model):
    for p in model.parameters():
        p.requires_grad = True

def unfreeze_head_only(model):
    for n, p in model.named_parameters():
        if any(k in n for k in ["head", "fc", "classifier"]):
            p.requires_grad = True

def unfreeze_partial(model, model_key):
    partial_keywords = {
        "swin_t": ["head", "layers.3", "norm"],
        "vit_small": ["head", "blocks.11", "norm"],
    }
    keys = partial_keywords.get(model_key, ["head", "fc", "classifier"])
    for n, p in model.named_parameters():
        if any(k in n for k in keys):
            p.requires_grad = True

def setup_model_for_head_stage(model):
    freeze_all(model)
    unfreeze_head_only(model)

def setup_model_for_ft_stage(model, model_key):
    freeze_all(model)
    if MODEL_CONFIGS[model_key]["mode"] == "full":
        unfreeze_all(model)
    else:
        unfreeze_partial(model, model_key)

def trainable_params_count(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def total_params_count(model):
    return sum(p.numel() for p in model.parameters())

def model_size_mb(model):
    return total_params_count(model) * 4 / (1024 ** 2)

for mk in MODEL_CONFIGS.keys():
    m = build_model(mk)
    setup_model_for_head_stage(m)
    print(f"{mk}: total_params={total_params_count(m):,} | head_trainable={trainable_params_count(m):,} | approx_size_mb={model_size_mb(m):.2f}")
    del m
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# Cell 7
def run_epoch(model, loader, criterion, optimizer=None, train_mode=True):
    model.train(train_mode)
    total_loss = 0.0
    all_preds = []
    all_labels = []
    scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train_mode):
            with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
                outputs = model(images)
                loss = criterion(outputs, labels)

            if train_mode:
                scaler.scale(loss).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()

        preds = outputs.argmax(dim=1)
        total_loss += loss.item() * images.size(0)
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_labels.extend(labels.detach().cpu().numpy().tolist())

    epoch_loss = total_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds) * 100
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    return epoch_loss, epoch_acc, prec * 100, rec * 100, f1 * 100

def status_flag(history, epoch_idx, train_loss, train_acc, val_loss, val_acc, val_f1):
    gap = train_acc - val_acc
    if epoch_idx < 2:
        return "STABLE", gap
    prev = history[-1]
    unstable = (abs(val_acc - prev["val_acc"]) > 5.0) or (abs(val_f1 - prev["val_f1"]) > 5.0)
    overfit = (gap > 6.0) and (val_loss > prev["val_loss"]) and (val_f1 < prev["val_f1"])
    underfit = (epoch_idx >= 2) and (train_acc < 70.0) and (val_acc < 70.0) and (train_loss > 0.8)
    if overfit:
        return "OVERFIT_WARN", gap
    if unstable:
        return "UNSTABLE_WARN", gap
    if underfit:
        return "UNDERFIT_WARN", gap
    return "STABLE", gap

def fit_one_model(model_key):
    cfg = MODEL_CONFIGS[model_key]
    model = build_model(model_key).to(DEVICE)
    train_loader, valid_loader, test_loader = make_loaders(cfg["batch_size"])
    criterion = nn.CrossEntropyLoss()

    history = []
    best_score = -1
    best_epoch = -1
    best_state = None
    patience_left = PATIENCE

    setup_model_for_head_stage(model)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=cfg["head_lr"], weight_decay=1e-4)

    total_epochs = cfg["head_epochs"] + cfg["ft_epochs"]

    for epoch in range(total_epochs):
        if epoch == cfg["head_epochs"]:
            setup_model_for_ft_stage(model, model_key)
            optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=cfg["ft_lr"], weight_decay=1e-4)

        train_loss, train_acc, _, _, _ = run_epoch(model, train_loader, criterion, optimizer=optimizer, train_mode=True)
        val_loss, val_acc, val_prec, val_rec, val_f1 = run_epoch(model, valid_loader, criterion, optimizer=None, train_mode=False)

        status, gap = status_flag(history, epoch, train_loss, train_acc, val_loss, val_acc, val_f1)
        lr_now = optimizer.param_groups[0]["lr"]

        row = {"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": val_loss, "val_acc": val_acc, "val_prec": val_prec, "val_rec": val_rec, "val_f1": val_f1, "lr": lr_now, "gap": gap, "status": status}
        history.append(row)

        print(
            f"[{model_key}] Epoch [{epoch+1}/{total_epochs}] | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f} | "
            f"Val Macro F1: {val_f1:.2f} | LR: {lr_now:.6f} | "
            f"Gap: {gap:.2f} | Status: {status}"
        )

        if val_f1 > best_score:
            best_score = val_f1
            best_epoch = epoch + 1
            best_state = copy.deepcopy(model.state_dict())
            patience_left = PATIENCE
        else:
            patience_left -= 1

        if patience_left <= 0:
            print(f"[{model_key}] Early stopping triggered at epoch {epoch+1}")
            break

    ckpt_path = CHECKPOINT_ROOT / f"{model_key}_best.pth"
    torch.save(best_state, ckpt_path)
    model.load_state_dict(best_state)

    y_true = []
    y_pred = []
    model.eval()
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE, non_blocking=True)
            outputs = model(images)
            preds = outputs.argmax(dim=1).cpu().numpy().tolist()
            y_pred.extend(preds)
            y_true.extend(labels.numpy().tolist())

    test_acc = accuracy_score(y_true, y_pred) * 100
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    cls_report = classification_report(y_true, y_pred, target_names=[idx_to_class[i] for i in sorted(idx_to_class.keys())], output_dict=True, zero_division=0)

    return {
        "model": model_key,
        "best_epoch": best_epoch,
        "test_accuracy": test_acc,
        "test_precision": p_macro * 100,
        "test_recall": r_macro * 100,
        "test_f1": f1_macro * 100,
        "test_macro_f1": f1_macro * 100,
        "params_m": total_params_count(model) / 1e6,
        "size_mb": model_size_mb(model),
        "checkpoint": str(ckpt_path),
        "history": history,
        "y_true": y_true,
        "y_pred": y_pred,
        "classification_report": cls_report,
    }

In [ ]:
# Cell 8
all_results = []

for model_key in MODEL_CONFIGS.keys():
    print("\n" + "=" * 100)
    print("Training model:", model_key)
    print("=" * 100)
    result = fit_one_model(model_key)
    all_results.append(result)
    print(
        f"\n[{model_key}] Best epoch: {result['best_epoch']} | "
        f"Test Acc: {result['test_accuracy']:.2f} | "
        f"Macro F1: {result['test_macro_f1']:.2f}"
    )
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# Cell 9
def measure_latency(model_key, ckpt_path, n_images=128):
    cfg = MODEL_CONFIGS[model_key]
    model = build_model(model_key).to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()

    _, _, test_loader = make_loaders(cfg["batch_size"])
    images_seen = 0
    times = []

    with torch.no_grad():
        for images, _ in test_loader:
            images = images.to(DEVICE, non_blocking=True)
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            start = time.perf_counter()
            _ = model(images)
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            end = time.perf_counter()
            times.append((end - start) / images.size(0))
            images_seen += images.size(0)
            if images_seen >= n_images:
                break

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return float(np.mean(times) * 1000.0)

for r in all_results:
    r["latency_ms_per_image"] = measure_latency(r["model"], r["checkpoint"])

overall_df = pd.DataFrame([{
    "Model": r["model"],
    "Accuracy": round(r["test_accuracy"], 2),
    "Precision": round(r["test_precision"], 2),
    "Recall": round(r["test_recall"], 2),
    "F1-score": round(r["test_f1"], 2),
    "Macro F1": round(r["test_macro_f1"], 2),
    "Params (M)": round(r["params_m"], 2),
    "Size (MB)": round(r["size_mb"], 2),
    "Latency (ms/img)": round(r["latency_ms_per_image"], 4),
    "Best Epoch": r["best_epoch"],
} for r in all_results])

display(overall_df)

per_class_rows = []
for r in all_results:
    for cls_name in sorted(base_train_dataset.class_to_idx.keys()):
        stats = r["classification_report"].get(cls_name, None)
        if stats is None:
            continue
        per_class_rows.append({
            "Class": cls_name,
            "Model": r["model"],
            "Accuracy": round(r["test_accuracy"], 2),
            "Precision": round(stats["precision"] * 100, 2),
            "Recall": round(stats["recall"] * 100, 2),
            "F1-score": round(stats["f1-score"] * 100, 2),
            "Macro F1": round(r["test_macro_f1"], 2),
        })

per_class_df = pd.DataFrame(per_class_rows)
display(per_class_df)

def inverse_minmax(series):
    s = series.astype(float)
    return (s.max() - s) / (s.max() - s.min() + 1e-12)

def forward_minmax(series):
    s = series.astype(float)
    return (s - s.min()) / (s.max() - s.min() + 1e-12)

overall_df["eff_params"] = inverse_minmax(overall_df["Params (M)"])
overall_df["eff_size"] = inverse_minmax(overall_df["Size (MB)"])
overall_df["eff_latency"] = inverse_minmax(overall_df["Latency (ms/img)"])
overall_df["Efficiency Score"] = (overall_df["eff_params"] + overall_df["eff_size"] + overall_df["eff_latency"]) / 3.0
overall_df["MacroF1_norm"] = forward_minmax(overall_df["Macro F1"])
overall_df["Trade-off Score"] = 0.6 * overall_df["MacroF1_norm"] + 0.4 * overall_df["Efficiency Score"]

display(overall_df[["Model", "Accuracy", "Precision", "Recall", "F1-score", "Macro F1", "Params (M)", "Size (MB)", "Latency (ms/img)", "Efficiency Score", "Trade-off Score"]])

best_perf_model = overall_df.sort_values("Macro F1", ascending=False).iloc[0]["Model"]
best_eff_model = overall_df.sort_values("Efficiency Score", ascending=False).iloc[0]["Model"]
best_trade_model = overall_df.sort_values("Trade-off Score", ascending=False).iloc[0]["Model"]

print("Best-performing model:", best_perf_model)
print("Best-efficient model:", best_eff_model)
print("Best trade-off model:", best_trade_model)

In [ ]:
# Cell 10
def plot_history(model_key):
    record = next(r for r in all_results if r["model"] == model_key)
    hist = pd.DataFrame(record["history"])

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    axes[0].plot(hist["epoch"], hist["train_loss"], label="Train Loss")
    axes[0].plot(hist["epoch"], hist["val_loss"], label="Val Loss")
    axes[0].set_title(f"{model_key} Loss Curve")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(hist["epoch"], hist["train_acc"], label="Train Acc")
    axes[1].plot(hist["epoch"], hist["val_acc"], label="Val Acc")
    axes[1].set_title(f"{model_key} Accuracy Curve")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    axes[2].plot(hist["epoch"], hist["val_f1"], label="Val Macro F1")
    axes[2].set_title(f"{model_key} Macro F1 Curve")
    axes[2].set_xlabel("Epoch")
    axes[2].legend()

    plt.tight_layout()
    plt.show()

def plot_confusion(model_key):
    record = next(r for r in all_results if r["model"] == model_key)
    labels = [idx_to_class[i] for i in sorted(idx_to_class.keys())]
    cm = confusion_matrix(record["y_true"], record["y_pred"])

    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.title(f"Confusion Matrix - {model_key}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

plot_history(best_perf_model)
plot_confusion(best_perf_model)

if best_eff_model != best_perf_model:
    plot_confusion(best_eff_model)

In [ ]:
# Cell 11
def get_target_layer(model_key, model):
    if model_key == "resnet50":
        return [model.layer4[-1]]
    if model_key == "densenet121":
        return [model.features.norm5]
    if model_key == "mobilenetv3_large":
        return [model.conv_head]
    if model_key == "efficientnet_b0":
        return [model.conv_head]
    if model_key == "swin_t":
        return [model.layers[-1].blocks[-1].norm1]
    if model_key == "vit_small":
        return [model.blocks[-1].norm1]
    raise ValueError("Unknown model key")

def vit_reshape_transform(tensor, height=14, width=14):
    result = tensor[:, 1:, :].reshape(tensor.size(0), height, width, tensor.size(2))
    result = result.permute(0, 3, 1, 2)
    return result

def swin_reshape_transform(tensor, height=7, width=7):
    result = tensor.reshape(tensor.size(0), height, width, tensor.size(2))
    result = result.permute(0, 3, 1, 2)
    return result

def get_reshape_transform(model_key):
    if model_key == "vit_small":
        return vit_reshape_transform
    if model_key == "swin_t":
        return swin_reshape_transform
    return None

best_record = next(r for r in all_results if r["model"] == best_perf_model)
best_model = build_model(best_perf_model).to(DEVICE)
best_model.load_state_dict(torch.load(best_record["checkpoint"], map_location=DEVICE))
best_model.eval()

target_layers = get_target_layer(best_perf_model, best_model)
reshape_transform = get_reshape_transform(best_perf_model)
cam = GradCAM(model=best_model, target_layers=target_layers, reshape_transform=reshape_transform)

samples_for_cam = []
class_seen = set()
for img_path, class_idx in test_dataset.samples:
    cls_name = idx_to_class[class_idx]
    if cls_name not in class_seen:
        samples_for_cam.append((img_path, class_idx))
        class_seen.add(cls_name)
    if len(class_seen) == NUM_CLASSES:
        break

fig, axes = plt.subplots(len(samples_for_cam), 3, figsize=(12, 4 * len(samples_for_cam)))
if len(samples_for_cam) == 1:
    axes = np.expand_dims(axes, axis=0)

for row_idx, (img_path, class_idx) in enumerate(samples_for_cam):
    pil_img = Image.open(img_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    input_tensor = eval_transform(pil_img).unsqueeze(0).to(DEVICE)

    grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(class_idx)])[0]

    img_np = np.array(pil_img).astype(np.float32) / 255.0
    heatmap = plt.cm.jet(grayscale_cam)[:, :, :3]
    overlay = np.clip(0.5 * img_np + 0.5 * heatmap, 0, 1)

    with torch.no_grad():
        pred = best_model(input_tensor).argmax(dim=1).item()

    axes[row_idx, 0].imshow(img_np)
    axes[row_idx, 0].set_title(f"Original\n{idx_to_class[class_idx]}")
    axes[row_idx, 0].axis("off")

    axes[row_idx, 1].imshow(grayscale_cam, cmap="jet")
    axes[row_idx, 1].set_title("Grad-CAM")
    axes[row_idx, 1].axis("off")

    axes[row_idx, 2].imshow(overlay)
    axes[row_idx, 2].set_title(f"Overlay\nPred: {idx_to_class[pred]}")
    axes[row_idx, 2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 12
print("Final outputs to screenshot for review:")
print("1. Before duplicate removal class distribution")
print("2. After duplicate removal class distribution")
print("3. Split distribution table")
print("4. Effective class balance after augmentation target")
print("5. Last 5-8 epoch logs of any model you want me to check")
print("6. Loss curve")
print("7. Accuracy curve")
print("8. Confusion matrix")
print("9. Per-class performance table")
print("10. Overall metrics / efficiency table")